In [ ]:
# Uncomment kode di bawah ini jika terdapat masalah dengan Java di Windows

# import os
# import sys

# JAVA = r"C:\path\to\your\java\jdk" # Ganti dengan path JDK yang sesuai

# os.environ["JAVA_HOME"] = JAVA
# os.environ["PATH"] = os.path.join(JAVA, "bin") + os.path.pathsep + os.environ.get("PATH", "")
# os.environ["PYSPARK_PYTHON"] = sys.executable
# os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

In [ ]:
%pip install -r requirements.txt
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, length, trim, lower

from pyspark.ml import Pipeline
from pyspark.ml.classification import LinearSVC
from pyspark.ml.feature import (
    IDF,
    HashingTF,
    Normalizer,
    StringIndexer,
    Tokenizer,
    IndexToString
)
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

os.makedirs("D:/spark-temp", exist_ok=True)

spark = SparkSession.builder \
    .appName("AmazonReviewsPolarity") \
    .config("spark.driver.memory", "4g") \
    .master("local[*]") \
    .config("spark.local.dir", "D:/spark-temp") \
    .getOrCreate()

df_train = spark.read.csv("../datasets/train.csv", header=True, inferSchema=True)
df_train = df_train.toDF("label", "title", "text")
df_train.show(5)

df_test = spark.read.csv("../datasets/test.csv", header=True, inferSchema=True)
df_test = df_test.toDF("label", "title", "text")
df_test.show(5)

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


+-----+--------------------+--------------------+
|label|               title|                text|
+-----+--------------------+--------------------+
|    2|The best soundtra...|I'm reading a lot...|
|    2|            Amazing!|"This soundtrack ...|
|    2|Excellent Soundtrack|I truly like this...|
|    2|Remember, Pull Yo...|If you've played ...|
|    2|an absolute maste...|I am quite sure a...|
+-----+--------------------+--------------------+
only showing top 5 rows
+-----+--------------------+--------------------+
|label|               title|                text|
+-----+--------------------+--------------------+
|    2|One of the best g...|Despite the fact ...|
|    1|Batteries died wi...|I bought this cha...|
|    2|works fine, but M...|Check out Maha En...|
|    2|Great for the non...|Reviewed quite a ...|
|    1|DVD Player crappe...|I also began havi...|
+-----+--------------------+--------------------+
only showing top 5 rows


In [34]:
def preprocessing(df):
    df_clean = df.dropna()
    
    df_clean = df_clean.dropDuplicates(["title", "text"])
    
    df_clean = df_clean \
        .withColumn("title", trim(col("title"))) \
        .withColumn("text", trim(col("text")))
    
    invalid_words = ["n/a", "null", "undefined", "-"]
    
    df_clean = df_clean.filter(
        col("label").isin(1, 2) &
        (length(col("title")) >= 3) &
        (length(col("text")) >= 3) &
        (~lower(col("text")).isin(invalid_words))
    )
    
    return df_clean

In [35]:

df_train = preprocessing(df_train)
df_test = preprocessing(df_test)

tokenizer = Tokenizer(inputCol="text", outputCol="words")

hashingTF = HashingTF(
    inputCol="words", outputCol="rawFeatures", numFeatures=10000
)

idf = IDF(inputCol="rawFeatures", outputCol="idfFeatures")

normalizer = Normalizer(inputCol="idfFeatures", outputCol="features", p=2.0)

label_indexer = StringIndexer(inputCol="label", outputCol="label_index")

svm = LinearSVC(featuresCol="features", labelCol="label_index", maxIter=15)

label_converter = IndexToString(
    inputCol="prediction",
    outputCol="predicted_label",
    labels=label_indexer.fit(df_train).labels,
)

pipeline = Pipeline(
    stages=[tokenizer, hashingTF, idf, normalizer, label_indexer, svm, label_converter]
)

model = pipeline.fit(df_train)

predictions = model.transform(df_test)

predictions.select("text", "label", "predicted_label").show(100)

+--------------------+-----+---------------+
|                text|label|predicted_label|
+--------------------+-----+---------------+
|I purchased a twi...|    2|              2|
|I used it once an...|    1|              1|
|I have used this ...|    1|              1|
|movies are okay b...|    1|              1|
|I was in 9th grad...|    2|              2|
|"Clay's voice soa...|    2|              2|
|This movie is ver...|    2|              2|
|I bought this fro...|    2|              1|
|This movie isnt e...|    1|              1|
|"This review rela...|    2|              2|
|"The character wa...|    1|              1|
|I must admit that...|    1|              1|
|be careful, when ...|    1|              1|
|According to this...|    1|              1|
|I found this vers...|    1|              1|
|"I had seen this ...|    2|              1|
|Overall a very go...|    2|              2|
|Another business ...|    1|              2|
|I have not heard ...|    2|              1|
|I am not 

In [38]:
evaluator = MulticlassClassificationEvaluator(
    labelCol="label_index",
    predictionCol="prediction",
)

accuracy = evaluator.setMetricName("accuracy").evaluate(predictions)
f1 = evaluator.setMetricName("f1").evaluate(predictions)
precision = evaluator.setMetricName("weightedPrecision").evaluate(predictions)
recall = evaluator.setMetricName("weightedRecall").evaluate(predictions)

print(f"Accuracy  : {accuracy:.4f}")
print(f"F1-Score  : {f1:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")

Accuracy  : 0.8373
F1-Score  : 0.8373
Precision : 0.8373
Recall    : 0.8373


In [40]:
predictions.groupBy(
    "label",
    "predicted_label"
).count().orderBy(
    "label",
    "predicted_label"
).show()

+-----+---------------+------+
|label|predicted_label| count|
+-----+---------------+------+
|    1|              1|167524|
|    1|              2| 32144|
|    2|              1| 32866|
|    2|              2|166973|
+-----+---------------+------+

